# Extractor

This will create a pickle file which contains the info needed for the learner and theory evaluator.

I don't think I need to add prolog/clingo clauses... that happens in the `amati_bea.py` program.

In [1]:
import pandas as pd

from pyrsistent import pmap

from itertools import count
from string import whitespace
import pickle

from compound_split import char_split, doc_split
import phunspell

## Part 1: load in the data

In [2]:
with open(
    '/Users/alistair.willis/repos/bea-2026/BEA Shared Task 2026 on Rubric-based Short Answer Scoring for German/updated/3way/ALICE_LP_train_3way__v2.json'
) as fIn:
    j1_df = pd.read_json(fIn)

j1_df["partition"] = "train"

j1_df.head()

,id,question_id,question,answer,rubric,score,partition
0,2a5d7d84-fd00-4cb1-859c-246604d8e522,bc2d01c9-424c-428a-9909-7114819a31d9,Beurteile das Zerteilen der Äpfel hinsichlich ...,Bei einem höheren Zerteilungsgrad ( hier das Z...,{'Incorrect': 'Die Schüler:innen beurteilen di...,Incorrect,train
1,9060ce69-53d7-4e47-8237-c939d11e9ef2,bc2d01c9-424c-428a-9909-7114819a31d9,Beurteile das Zerteilen der Äpfel hinsichlich ...,"Das Größte apfelstück , also das mildem gering...",{'Incorrect': 'Die Schüler:innen beurteilen di...,Correct,train
2,c5a42d7e-ae48-4c5b-87a3-2db32bbbb773,bc2d01c9-424c-428a-9909-7114819a31d9,Beurteile das Zerteilen der Äpfel hinsichlich ...,verderben schneller\n,{'Incorrect': 'Die Schüler:innen beurteilen di...,Incorrect,train
3,8207b3a1-675e-42c9-921f-e745cbcdf5f0,bc2d01c9-424c-428a-9909-7114819a31d9,Beurteile das Zerteilen der Äpfel hinsichlich ...,Durch zerteilen der Äpfel wird dessen oberfläc...,{'Incorrect': 'Die Schüler:innen beurteilen di...,Correct,train
4,7c2e3e09-c34a-44a6-aa43-fb614fbcd129,bc2d01c9-424c-428a-9909-7114819a31d9,Beurteile das Zerteilen der Äpfel hinsichlich ...,"Je größer die Oberfläche vom Apfel ist , desto...",{'Incorrect': 'Die Schüler:innen beurteilen di...,Partially correct,train


In [3]:
with open(
    "/Users/alistair.willis/repos/bea-2026/BEA Shared Task 2026 on Rubric-based Short Answer Scoring for German/updated/3way/ALICE_LP_trial_3way__v2.json"
) as fIn:
    j2_df = pd.read_json(fIn)

j2_df["partition"] = "trial"

j2_df.head()

,id,question_id,question,answer,rubric,score,partition
0,36646661-efef-4070-bcce-87273ca9a82b,bc2d01c9-424c-428a-9909-7114819a31d9,Beurteile das Zerteilen der Äpfel hinsichlich ...,Durch Zerteilen der Äpfel ändert sich das Verh...,{'Incorrect': 'Die Schüler:innen beurteilen di...,Incorrect,trial
1,962ea355-6849-46b5-ab50-c74e7c98da50,bc2d01c9-424c-428a-9909-7114819a31d9,Beurteile das Zerteilen der Äpfel hinsichlich ...,Zerteilter Apfel = höherer Zerteilungsgrad = h...,{'Incorrect': 'Die Schüler:innen beurteilen di...,Correct,trial
2,4f7f3884-963e-4c46-8458-a84c6014782e,bc2d01c9-424c-428a-9909-7114819a31d9,Beurteile das Zerteilen der Äpfel hinsichlich ...,"Je öfter der Apfel zerteilt wurde , desto höhe...",{'Incorrect': 'Die Schüler:innen beurteilen di...,Correct,trial
3,27bc7e48-c8d6-4c2d-a51d-9a28864eaab3,bc2d01c9-424c-428a-9909-7114819a31d9,Beurteile das Zerteilen der Äpfel hinsichlich ...,äpfel verderben schneller bei höheren temperat...,{'Incorrect': 'Die Schüler:innen beurteilen di...,Incorrect,trial
4,8cd711d0-a0be-46af-832a-e9c30cc6ae03,bc2d01c9-424c-428a-9909-7114819a31d9,Beurteile das Zerteilen der Äpfel hinsichlich ...,"Je kleiner die Äpfel geschnitten werden , dest...",{'Incorrect': 'Die Schüler:innen beurteilen di...,Correct,trial


In [4]:
j_df = pd.concat([j1_df, j2_df], ignore_index=True)

j_df

,id,question_id,question,answer,rubric,score,partition
0,2a5d7d84-fd00-4cb1-859c-246604d8e522,bc2d01c9-424c-428a-9909-7114819a31d9,Beurteile das Zerteilen der Äpfel hinsichlich ...,Bei einem höheren Zerteilungsgrad ( hier das Z...,{'Incorrect': 'Die Schüler:innen beurteilen di...,Incorrect,train
1,9060ce69-53d7-4e47-8237-c939d11e9ef2,bc2d01c9-424c-428a-9909-7114819a31d9,Beurteile das Zerteilen der Äpfel hinsichlich ...,"Das Größte apfelstück , also das mildem gering...",{'Incorrect': 'Die Schüler:innen beurteilen di...,Correct,train
2,c5a42d7e-ae48-4c5b-87a3-2db32bbbb773,bc2d01c9-424c-428a-9909-7114819a31d9,Beurteile das Zerteilen der Äpfel hinsichlich ...,verderben schneller\n,{'Incorrect': 'Die Schüler:innen beurteilen di...,Incorrect,train
3,8207b3a1-675e-42c9-921f-e745cbcdf5f0,bc2d01c9-424c-428a-9909-7114819a31d9,Beurteile das Zerteilen der Äpfel hinsichlich ...,Durch zerteilen der Äpfel wird dessen oberfläc...,{'Incorrect': 'Die Schüler:innen beurteilen di...,Correct,train
4,7c2e3e09-c34a-44a6-aa43-fb614fbcd129,bc2d01c9-424c-428a-9909-7114819a31d9,Beurteile das Zerteilen der Äpfel hinsichlich ...,"Je größer die Oberfläche vom Apfel ist , desto...",{'Incorrect': 'Die Schüler:innen beurteilen di...,Partially correct,train
...,...,...,...,...,...,...,...
7894,50ebd5ae-eb16-4284-b22e-ae3169e42058,67c1649e-b694-4813-be49-abfd65a6798e,Beschreibe Auffälligkeiten der dargestellten d...,Nicht bei jeder Kollision sind die Edukte zu e...,{'Incorrect': 'Die Schüler:innen beschreiben d...,Correct,trial
7895,0f922c54-3c13-4683-b7a6-353588b79677,67c1649e-b694-4813-be49-abfd65a6798e,Beschreibe Auffälligkeiten der dargestellten d...,Nicht alle Kollisionen der Edukt-Teilchen führ...,{'Incorrect': 'Die Schüler:innen beschreiben d...,Correct,trial
7896,483689d8-0601-4e4f-9971-53803def8af6,67c1649e-b694-4813-be49-abfd65a6798e,Beschreibe Auffälligkeiten der dargestellten d...,Hab ich doch in Aufgabe 2 gemacht\n,{'Incorrect': 'Die Schüler:innen beschreiben d...,Incorrect,trial
7897,784320e2-1809-4514-a675-a2fc1b230aff,67c1649e-b694-4813-be49-abfd65a6798e,Beschreibe Auffälligkeiten der dargestellten d...,Nicht immer führt eine Kollision zu einer Reak...,{'Incorrect': 'Die Schüler:innen beschreiben d...,Partially correct,trial


In [5]:
j_df['rubric'].apply(tuple).drop_duplicates()

0    (Incorrect, Partially correct, Correct)
Name: rubric, dtype: object

There's a `Correct`, `Incorrect` and `Partially correct` rubric for each question.

Let's tidy things up a bit so the ids are more readable.

In [6]:
question_ids = {
    q_id: f"q_{str(q1).zfill(2)}"
    for (q_id, q1) in zip(set(j_df["question_id"]), count())
}

question_ids

{'1f3469d5-f6a8-49f4-85d0-d2e156e4cb0c': 'q_00',
 '75333f3a-c1c2-4729-8d61-146800f16532': 'q_01',
 'dfeace94-8239-4f80-b769-1e0bebbc391f': 'q_02',
 '10d57539-7664-4f0f-afe2-51a749036f54': 'q_03',
 '4a79233b-2d16-4c6d-ba9f-53a7bedf49fc': 'q_04',
 'aa16fbaa-d192-46b4-b338-395bfc6bc58b': 'q_05',
 '807991c9-2b51-4c91-88f7-86b0cbc129e3': 'q_06',
 '5c0edfac-2536-455a-8078-3adceeecfa43': 'q_07',
 '838b4e27-f33d-439d-987b-57ddece063ad': 'q_08',
 'afe4a36a-bbce-40da-a70c-03158b57b6cb': 'q_09',
 'c7a7a954-e315-4ec2-90df-48e5a0940e25': 'q_10',
 '4ba62250-a272-4b8f-bf28-eb531661fcc8': 'q_11',
 'da9e0a6a-7fd4-47ad-aa6a-a0f91fda560e': 'q_12',
 '4e3a3e06-40c4-479b-8882-36005aae1e4e': 'q_13',
 'b845b9f0-92af-4805-be05-b8faf5960dd1': 'q_14',
 '33385739-7e9d-4aa6-9118-62e6772f7eb1': 'q_15',
 '4a38efeb-0a3c-4ff1-8320-2197abb410a3': 'q_16',
 'f38d01cf-f8e1-4e68-a11f-162b9cb6f0c7': 'q_17',
 '64393e93-5585-42f9-ad22-ffa71219573b': 'q_18',
 '741376db-a5f7-43a1-abc4-988ff40cf642': 'q_19',
 'e9e8d257-f86c-4ac9

In [7]:
response_ids = {
    r_id: f"r_{str(r1).zfill(4)}" for (r_id, r1) in zip(set(j_df["id"]), count())
}

response_ids

{'4af8c054-8dd8-4333-aa1a-c5811bb117c5': 'r_0000',
 '25ee90a6-f0dc-4367-9dcf-a8a26a4ff856': 'r_0001',
 '06de3578-9de7-43a9-9d70-bd6ad41684af': 'r_0002',
 '9f9c3b10-8d37-49d4-9e53-8a66067ffeee': 'r_0003',
 '6dacc962-fb14-4022-beb0-46f167bced0c': 'r_0004',
 '7460d293-b13a-4ff5-8cb9-f19cc78d28ff': 'r_0005',
 '57753aa2-cd69-49ff-b6f9-bd2869eaab57': 'r_0006',
 '84f30c8a-9238-4ab6-aa1e-6a402f45e30c': 'r_0007',
 'a8ec06e5-6e7f-472d-8a55-6681642e59ec': 'r_0008',
 '5ead86aa-f1d7-48bc-bd8c-a3cea79c24a0': 'r_0009',
 'eb827966-2bc5-4b6a-b1d7-3ab688d22d05': 'r_0010',
 '5b082d98-ac17-4162-9907-1404fa4925e3': 'r_0011',
 '02295bcf-1718-4644-9c04-393205285b2f': 'r_0012',
 'eddcdbf0-b2e8-48e1-a2df-061b9a39d969': 'r_0013',
 '61c4de77-ad4f-41af-9ef7-f334a786aebe': 'r_0014',
 '20e8be9d-3114-45f4-bacf-8d84b29ef884': 'r_0015',
 '50b33ec3-0e3c-468e-bda9-263a2d762659': 'r_0016',
 '1701d0f1-9e3e-4b94-bdfd-ec630a5cbfec': 'r_0017',
 '1cc1dd01-7472-46ac-bae4-7e02e6a3e5fb': 'r_0018',
 '3b8e297e-e57d-42f1-878c-de514

In [8]:
id_mapper = pmap(question_ids).update(response_ids)
id_mapper["67c1649e-b694-4813-be49-abfd65a6798e"]

'q_62'

I'm going to convert the score to lower case as it'll make everything easier in clingo. Also need to replace 'partially correct' with 'partially_correct' so that there's a single atom.

In [9]:
# Note to students: don't write code like this!

bea_3way_data_df = (
    j_df.rename(
        {
            "id": "original_response_id",
            "question_id": "original_question_id",
            "answer": "response",
            "score": "capitalised_score",
        },
        axis="columns",
    )
    .assign(
        response_id=j_df["id"].map(id_mapper),
        question_id=j_df["question_id"].map(id_mapper),
        correct=j_df["rubric"].apply(lambda x: x["Correct"]),
        incorrect=j_df["rubric"].apply(lambda x: x["Incorrect"]),
        partially_correct=j_df["rubric"].apply(lambda x: x["Partially correct"]),
        score=j_df["score"].str.lower().str.replace('partially correct', 'partially_correct'),
    )
    .drop(["rubric", "capitalised_score"], axis="columns")
)

bea_3way_data_df

,original_response_id,original_question_id,question,response,partition,response_id,question_id,correct,incorrect,partially_correct,score
0,2a5d7d84-fd00-4cb1-859c-246604d8e522,bc2d01c9-424c-428a-9909-7114819a31d9,Beurteile das Zerteilen der Äpfel hinsichlich ...,Bei einem höheren Zerteilungsgrad ( hier das Z...,train,r_4856,q_31,Die Schüler:innen beurteilen die Verderblichke...,Die Schüler:innen beurteilen die Verderblichke...,Die Schüler:innen beurteilen die Verderblichke...,incorrect
1,9060ce69-53d7-4e47-8237-c939d11e9ef2,bc2d01c9-424c-428a-9909-7114819a31d9,Beurteile das Zerteilen der Äpfel hinsichlich ...,"Das Größte apfelstück , also das mildem gering...",train,r_3158,q_31,Die Schüler:innen beurteilen die Verderblichke...,Die Schüler:innen beurteilen die Verderblichke...,Die Schüler:innen beurteilen die Verderblichke...,correct
2,c5a42d7e-ae48-4c5b-87a3-2db32bbbb773,bc2d01c9-424c-428a-9909-7114819a31d9,Beurteile das Zerteilen der Äpfel hinsichlich ...,verderben schneller\n,train,r_1090,q_31,Die Schüler:innen beurteilen die Verderblichke...,Die Schüler:innen beurteilen die Verderblichke...,Die Schüler:innen beurteilen die Verderblichke...,incorrect
3,8207b3a1-675e-42c9-921f-e745cbcdf5f0,bc2d01c9-424c-428a-9909-7114819a31d9,Beurteile das Zerteilen der Äpfel hinsichlich ...,Durch zerteilen der Äpfel wird dessen oberfläc...,train,r_3214,q_31,Die Schüler:innen beurteilen die Verderblichke...,Die Schüler:innen beurteilen die Verderblichke...,Die Schüler:innen beurteilen die Verderblichke...,correct
4,7c2e3e09-c34a-44a6-aa43-fb614fbcd129,bc2d01c9-424c-428a-9909-7114819a31d9,Beurteile das Zerteilen der Äpfel hinsichlich ...,"Je größer die Oberfläche vom Apfel ist , desto...",train,r_6138,q_31,Die Schüler:innen beurteilen die Verderblichke...,Die Schüler:innen beurteilen die Verderblichke...,Die Schüler:innen beurteilen die Verderblichke...,partially_correct
...,...,...,...,...,...,...,...,...,...,...,...
7894,50ebd5ae-eb16-4284-b22e-ae3169e42058,67c1649e-b694-4813-be49-abfd65a6798e,Beschreibe Auffälligkeiten der dargestellten d...,Nicht bei jeder Kollision sind die Edukte zu e...,trial,r_6276,q_62,Die Schüler:innen beschreiben die dargestellte...,Die Schüler:innen beschreiben die dargestellte...,Die Schüler:innen beschreiben die dargstellte ...,correct
7895,0f922c54-3c13-4683-b7a6-353588b79677,67c1649e-b694-4813-be49-abfd65a6798e,Beschreibe Auffälligkeiten der dargestellten d...,Nicht alle Kollisionen der Edukt-Teilchen führ...,trial,r_7420,q_62,Die Schüler:innen beschreiben die dargestellte...,Die Schüler:innen beschreiben die dargestellte...,Die Schüler:innen beschreiben die dargstellte ...,correct
7896,483689d8-0601-4e4f-9971-53803def8af6,67c1649e-b694-4813-be49-abfd65a6798e,Beschreibe Auffälligkeiten der dargestellten d...,Hab ich doch in Aufgabe 2 gemacht\n,trial,r_4984,q_62,Die Schüler:innen beschreiben die dargestellte...,Die Schüler:innen beschreiben die dargestellte...,Die Schüler:innen beschreiben die dargstellte ...,incorrect
7897,784320e2-1809-4514-a675-a2fc1b230aff,67c1649e-b694-4813-be49-abfd65a6798e,Beschreibe Auffälligkeiten der dargestellten d...,Nicht immer führt eine Kollision zu einer Reak...,trial,r_0880,q_62,Die Schüler:innen beschreiben die dargestellte...,Die Schüler:innen beschreiben die dargestellte...,Die Schüler:innen beschreiben die dargstellte ...,partially_correct


In [10]:
questions_df = bea_3way_data_df[
    ["original_question_id",
        "question_id",
        "question",
        "correct",
        "partially_correct",
        "incorrect",
    ]
]

questions_df = questions_df.assign(
    correct_id=questions_df["question_id"].replace(r"q(.*)", r"c\1", regex=True),
    partially_correct_id=questions_df["question_id"].replace(r"q(.*)", r"p\1", regex=True),
    incorrect_id=questions_df["question_id"].replace(r"q(.*)", r"i\1", regex=True),
)

questions_df = questions_df.drop_duplicates().sort_values(
    "question_id", ignore_index=True
)
questions_df

,original_question_id,question_id,question,correct,partially_correct,incorrect,correct_id,partially_correct_id,incorrect_id
0,1f3469d5-f6a8-49f4-85d0-d2e156e4cb0c,q_00,"Erkläre kurz, warum du dich für wahr oder fals...",Die SuS beschreiben die Steigung des Graphen a...,Die SuS beschreiben entweder die Steigung des ...,Die SuS beschreiben weder die Steigung des Gra...,c_00,p_00,i_00
1,75333f3a-c1c2-4729-8d61-146800f16532,q_01,Beschreibe die oben dargstellten Phänomene in ...,Die Schüler:innen beschreiben den Ablauf der R...,Die Schüler:innen beschreiben den Ablauf der R...,Die Schüler:innen beschreiben weder die Reakti...,c_01,p_01,i_01
2,dfeace94-8239-4f80-b769-1e0bebbc391f,q_02,Begründe deine Auswahl.,Die Schüler:innen stellen einen Zusammenhang z...,Die Schüler:innen stellen einen Zusammenhang z...,Die Schüler:innen stellen keinen Zusammenhang ...,c_02,p_02,i_02
3,10d57539-7664-4f0f-afe2-51a749036f54,q_03,"Erläutere, warum du (Wahr) oder (Falsch) angek...",Die SuS beschreiben die Unabhängigkeit des Ans...,Die SuS beschreiben entweder die Unabhängigkei...,Die SuS beschreiben weder die Unabhängigkeit d...,c_03,p_03,i_03
4,4a79233b-2d16-4c6d-ba9f-53a7bedf49fc,q_04,Es wird jeweils nur ein Teil der Eingangsleist...,"Die SuS beschreiben, dass die restliche Energi...","Die SuS nutzen Umwandlung, ohne auf die thermi...",Die SuS beschreiben die Umwandlung (in thermis...,c_04,p_04,i_04
...,...,...,...,...,...,...,...,...,...
73,442a4171-9f3a-40b2-b4b5-7c023bbfd4c4,q_73,Formuliere selbst eine Fragestellung.,Die SuS beschreiben die durchschnittliche Gesc...,Die SuS beschreiben entweder die durchschnittl...,Die SuS beschreiben weder die durchschnittlich...,c_73,p_73,i_73
74,2ad4c4b0-5dfe-4667-b344-a00434eeccf1,q_74,Stelle eine Hypothese zum Einfluss der Tempera...,Die Hypothese stellt eine überprüfbare Vermutu...,Die Hypothese stellt eine überprüfbare Vermutu...,Die Hypothese stellt keine überprüfbare Vermut...,c_74,p_74,i_74
75,9a2e7fec-b8fe-49d5-853c-2c0a32c36285,q_75,Wie strukturierst Du den Versuchsablauf? Besch...,Die Schüler:innen beschreiben und begründen ei...,Die Schüler:innen beschreiben eine systematisc...,Die Schüler:innen beschreiben und begründen ei...,c_75,p_75,i_75
76,ba753563-faa3-43c6-973d-320e5adc6979,q_76,Öffne das Material für deine Gruppe! Schaue di...,Die SuS beschreiben korrekt ...\nwie sich die ...,Die SuS beschreiben korrekt ...\nwie sich die ...,Die SuS beschreiben die Daten aus beiden Becke...,c_76,p_76,i_76


In [11]:
responses_df = bea_3way_data_df[
    ["original_response_id", "response_id", "question_id", "response", "score", "partition"]
].sort_values(["question_id", "partition", "response_id"], ignore_index=True)


responses_df

,original_response_id,response_id,question_id,response,score,partition
0,4af8c054-8dd8-4333-aa1a-c5811bb117c5,r_0000,q_00,"Ich habe mich für Ja entschieden , weil man ja...",incorrect,train
1,25ee90a6-f0dc-4367-9dcf-a8a26a4ff856,r_0001,q_00,"Ich habe mich für wahr entschieden , weil die ...",partially_correct,train
2,27536134-a7e3-4162-9d48-a1bec7cfb3e6,r_0098,q_00,Man kann es mit einer Formel berechnen\n,incorrect,train
3,b5c71e85-b027-4fb4-8f3e-fd005483f8a6,r_0114,q_00,"Ich habe mich für wahr entschieden , weil man ...",partially_correct,train
4,d6180f6e-c0b2-4e28-9d02-a48375ac7114,r_0134,q_00,"Ich habe mich für wahr entschieden , weil man ...",partially_correct,train
...,...,...,...,...,...,...
7894,e5eaa402-742d-4adc-9514-adde1b84ea59,r_3908,q_77,Beantwortung der Impulsfrage Mögliche Darstell...,incorrect,trial
7895,7c1f732f-9dc8-4db7-960a-de57f101550d,r_5695,q_77,Beantwortung der Impulsfrage Mögliche Darstell...,incorrect,trial
7896,dc27d04b-87b9-4d8c-b7f1-77664c043f03,r_5893,q_77,Beantwortung der Impulsfrage Mögliche Darstell...,correct,trial
7897,87ee9a44-34cc-4b0e-a60b-3a92681eb686,r_7192,q_77,Beantwortung der Impulsfrage Mögliche Darstell...,incorrect,trial


## Part 2: Extract lemmata with SpaCy

We will use SpaCy to convert the sentences into a usable form.

In [12]:
import spacy
import contextualSpellCheck

In [13]:
nlp = spacy.load("de_core_news_md")

# The spell check is causing issues at the moment. I'll return to this later...
# contextualSpellCheck.add_to_pipe(nlp)

In [14]:
de = nlp("junge reicht auch langsam mal")
de

junge reicht auch langsam mal

Don't need to be particularly finnicky with this in the first instance. We have essentially four types of sentence: question, correct answer, incorrect "answer", and response. We've indexed them differently, so we can just (?) bung them all into a big DataFrame:

In [15]:
# Again, stern note to students: don't write code like this!!

text_df = pd.concat(
    [
        questions_df[["question_id", "question"]].rename(
            {"question_id": "id", "question": "text"}, axis="columns"
        ),
        questions_df[["correct_id", "correct"]].rename(
            {"correct": "text", "correct_id": "id"}, axis="columns"
        ),
        questions_df[["partially_correct_id", "partially_correct"]].rename(
            {"partially_correct": "text", "partially_correct_id": "id"}, axis="columns"
        ),
        questions_df[["incorrect_id", "incorrect"]].rename(
            {"incorrect": "text", "incorrect_id": "id"}, axis="columns"
        ),
        responses_df[["response_id", "response"]].rename(
            {"response_id": "id", "response": "text"}, axis="columns"
        ),
    ],
    ignore_index=True,
)

text_df

,id,text
0,q_00,"Erkläre kurz, warum du dich für wahr oder fals..."
1,q_01,Beschreibe die oben dargstellten Phänomene in ...
2,q_02,Begründe deine Auswahl.
3,q_03,"Erläutere, warum du (Wahr) oder (Falsch) angek..."
4,q_04,Es wird jeweils nur ein Teil der Eingangsleist...
...,...,...
8206,r_3908,Beantwortung der Impulsfrage Mögliche Darstell...
8207,r_5695,Beantwortung der Impulsfrage Mögliche Darstell...
8208,r_5893,Beantwortung der Impulsfrage Mögliche Darstell...
8209,r_7192,Beantwortung der Impulsfrage Mögliche Darstell...


OK, so now we can apply SpaCy to each of the texts to get the appropriate data:

In [16]:
text_df = text_df.assign(spacy=text_df["text"].apply(nlp))
text_df

,id,text,spacy
0,q_00,"Erkläre kurz, warum du dich für wahr oder fals...","(Erkläre, kurz, ,, warum, du, dich, für, wahr,..."
1,q_01,Beschreibe die oben dargstellten Phänomene in ...,"(Beschreibe, die, oben, dargstellten, Phänomen..."
2,q_02,Begründe deine Auswahl.,"(Begründe, deine, Auswahl, .)"
3,q_03,"Erläutere, warum du (Wahr) oder (Falsch) angek...","(Erläutere, ,, warum, du, (, Wahr, ), oder, (,..."
4,q_04,Es wird jeweils nur ein Teil der Eingangsleist...,"(Es, wird, jeweils, nur, ein, Teil, der, Einga..."
...,...,...,...
8206,r_3908,Beantwortung der Impulsfrage Mögliche Darstell...,"(Beantwortung, der, Impulsfrage, Mögliche, Dar..."
8207,r_5695,Beantwortung der Impulsfrage Mögliche Darstell...,"(Beantwortung, der, Impulsfrage, Mögliche, Dar..."
8208,r_5893,Beantwortung der Impulsfrage Mögliche Darstell...,"(Beantwortung, der, Impulsfrage, Mögliche, Dar..."
8209,r_7192,Beantwortung der Impulsfrage Mögliche Darstell...,"(Beantwortung, der, Impulsfrage, Mögliche, Dar..."


I'm putting all the lemmata into lower case... I suspect this is naughty, but I'm going to see how it goes first.

In [17]:
text_df = text_df.explode("spacy", ignore_index=True)
text_df = text_df.assign(
    i=text_df["spacy"].apply(lambda x: x.i),
    lemma=text_df["spacy"].apply(lambda x: x.lemma_.lower()),
    word_text=text_df["spacy"].apply(lambda x: x.text),
)
text_df

,id,text,spacy,i,lemma,word_text
0,q_00,"Erkläre kurz, warum du dich für wahr oder fals...",Erkläre,0,erkläre,Erkläre
1,q_00,"Erkläre kurz, warum du dich für wahr oder fals...",kurz,1,kurz,kurz
2,q_00,"Erkläre kurz, warum du dich für wahr oder fals...",",",2,--,","
3,q_00,"Erkläre kurz, warum du dich für wahr oder fals...",warum,3,warum,warum
4,q_00,"Erkläre kurz, warum du dich für wahr oder fals...",du,4,du,du
...,...,...,...,...,...,...
275450,r_7715,Beantwortung der Impulsfrage Mögliche Darstell...,Energie,32,energie,Energie
275451,r_7715,Beantwortung der Impulsfrage Mögliche Darstell...,erreicht,33,erreichen,erreicht
275452,r_7715,Beantwortung der Impulsfrage Mögliche Darstell...,hat,34,haben,hat
275453,r_7715,Beantwortung der Impulsfrage Mögliche Darstell...,.,35,--,.


I think I'm also going to remove any cases where the lemma is just whitespace.

In [18]:
def whitespace_str(str_in):
    return all([c in whitespace for c in str_in])


assert whitespace_str(
    """    
                      
                                """
)
assert not whitespace_str("slkdjf  s")

In [19]:
text_df = text_df[-text_df["lemma"].apply(whitespace_str)]

-----------------------------

In [20]:
questions_df

,original_question_id,question_id,question,correct,partially_correct,incorrect,correct_id,partially_correct_id,incorrect_id
0,1f3469d5-f6a8-49f4-85d0-d2e156e4cb0c,q_00,"Erkläre kurz, warum du dich für wahr oder fals...",Die SuS beschreiben die Steigung des Graphen a...,Die SuS beschreiben entweder die Steigung des ...,Die SuS beschreiben weder die Steigung des Gra...,c_00,p_00,i_00
1,75333f3a-c1c2-4729-8d61-146800f16532,q_01,Beschreibe die oben dargstellten Phänomene in ...,Die Schüler:innen beschreiben den Ablauf der R...,Die Schüler:innen beschreiben den Ablauf der R...,Die Schüler:innen beschreiben weder die Reakti...,c_01,p_01,i_01
2,dfeace94-8239-4f80-b769-1e0bebbc391f,q_02,Begründe deine Auswahl.,Die Schüler:innen stellen einen Zusammenhang z...,Die Schüler:innen stellen einen Zusammenhang z...,Die Schüler:innen stellen keinen Zusammenhang ...,c_02,p_02,i_02
3,10d57539-7664-4f0f-afe2-51a749036f54,q_03,"Erläutere, warum du (Wahr) oder (Falsch) angek...",Die SuS beschreiben die Unabhängigkeit des Ans...,Die SuS beschreiben entweder die Unabhängigkei...,Die SuS beschreiben weder die Unabhängigkeit d...,c_03,p_03,i_03
4,4a79233b-2d16-4c6d-ba9f-53a7bedf49fc,q_04,Es wird jeweils nur ein Teil der Eingangsleist...,"Die SuS beschreiben, dass die restliche Energi...","Die SuS nutzen Umwandlung, ohne auf die thermi...",Die SuS beschreiben die Umwandlung (in thermis...,c_04,p_04,i_04
...,...,...,...,...,...,...,...,...,...
73,442a4171-9f3a-40b2-b4b5-7c023bbfd4c4,q_73,Formuliere selbst eine Fragestellung.,Die SuS beschreiben die durchschnittliche Gesc...,Die SuS beschreiben entweder die durchschnittl...,Die SuS beschreiben weder die durchschnittlich...,c_73,p_73,i_73
74,2ad4c4b0-5dfe-4667-b344-a00434eeccf1,q_74,Stelle eine Hypothese zum Einfluss der Tempera...,Die Hypothese stellt eine überprüfbare Vermutu...,Die Hypothese stellt eine überprüfbare Vermutu...,Die Hypothese stellt keine überprüfbare Vermut...,c_74,p_74,i_74
75,9a2e7fec-b8fe-49d5-853c-2c0a32c36285,q_75,Wie strukturierst Du den Versuchsablauf? Besch...,Die Schüler:innen beschreiben und begründen ei...,Die Schüler:innen beschreiben eine systematisc...,Die Schüler:innen beschreiben und begründen ei...,c_75,p_75,i_75
76,ba753563-faa3-43c6-973d-320e5adc6979,q_76,Öffne das Material für deine Gruppe! Schaue di...,Die SuS beschreiben korrekt ...\nwie sich die ...,Die SuS beschreiben korrekt ...\nwie sich die ...,Die SuS beschreiben die Daten aus beiden Becke...,c_76,p_76,i_76


In [21]:
responses_df

,original_response_id,response_id,question_id,response,score,partition
0,4af8c054-8dd8-4333-aa1a-c5811bb117c5,r_0000,q_00,"Ich habe mich für Ja entschieden , weil man ja...",incorrect,train
1,25ee90a6-f0dc-4367-9dcf-a8a26a4ff856,r_0001,q_00,"Ich habe mich für wahr entschieden , weil die ...",partially_correct,train
2,27536134-a7e3-4162-9d48-a1bec7cfb3e6,r_0098,q_00,Man kann es mit einer Formel berechnen\n,incorrect,train
3,b5c71e85-b027-4fb4-8f3e-fd005483f8a6,r_0114,q_00,"Ich habe mich für wahr entschieden , weil man ...",partially_correct,train
4,d6180f6e-c0b2-4e28-9d02-a48375ac7114,r_0134,q_00,"Ich habe mich für wahr entschieden , weil man ...",partially_correct,train
...,...,...,...,...,...,...
7894,e5eaa402-742d-4adc-9514-adde1b84ea59,r_3908,q_77,Beantwortung der Impulsfrage Mögliche Darstell...,incorrect,trial
7895,7c1f732f-9dc8-4db7-960a-de57f101550d,r_5695,q_77,Beantwortung der Impulsfrage Mögliche Darstell...,incorrect,trial
7896,dc27d04b-87b9-4d8c-b7f1-77664c043f03,r_5893,q_77,Beantwortung der Impulsfrage Mögliche Darstell...,correct,trial
7897,87ee9a44-34cc-4b0e-a60b-3a92681eb686,r_7192,q_77,Beantwortung der Impulsfrage Mögliche Darstell...,incorrect,trial


Quickly drop the `spacy` column, as that isn't picklable.

In [22]:
text_df.drop('spacy', axis='columns')

,id,text,i,lemma,word_text
0,q_00,"Erkläre kurz, warum du dich für wahr oder fals...",0,erkläre,Erkläre
1,q_00,"Erkläre kurz, warum du dich für wahr oder fals...",1,kurz,kurz
2,q_00,"Erkläre kurz, warum du dich für wahr oder fals...",2,--,","
3,q_00,"Erkläre kurz, warum du dich für wahr oder fals...",3,warum,warum
4,q_00,"Erkläre kurz, warum du dich für wahr oder fals...",4,du,du
...,...,...,...,...,...
275449,r_7715,Beantwortung der Impulsfrage Mögliche Darstell...,31,benötigt,benötigte
275450,r_7715,Beantwortung der Impulsfrage Mögliche Darstell...,32,energie,Energie
275451,r_7715,Beantwortung der Impulsfrage Mögliche Darstell...,33,erreichen,erreicht
275452,r_7715,Beantwortung der Impulsfrage Mögliche Darstell...,34,haben,hat


There's one other wrinkle: some of the lemmata contain quotation marks, which screws up the training file. As an interim hacky solution, I'll simply swap any instances of `"` for `'`. I think that the `"` appear in code, so it's not a sensible switch, but it'll do for the moment.

In [23]:
text_df=text_df.drop('lemma', axis='columns').assign(lemma=text_df['lemma'].str.replace('"', "'"))
                                                    
text_df.query("""
              `word_text`.str.contains('"')"""
              
              
              ).sort_values('word_text')

,id,text,spacy,i,word_text,lemma
149,q_10,Leitet auf Grundlage der Muster und der Messwe...,"""",9,"""",--
195735,r_3563,"Der Versuchsaufbau unterscheidet sich darin , ...","""",46,"""",--
196464,r_4175,Beide Reaktionsansätze sind fast gleich aufgeb...,"""",46,"""",--
196466,r_4175,Beide Reaktionsansätze sind fast gleich aufgeb...,"""",48,"""",--
201023,r_6949,"Je mehr Edukte es gibt , desto höher ist die C...","""",33,"""",--
...,...,...,...,...,...,...
273700,r_5135,Beantwortung der Impulsfrage Mögliche Darstell...,"id=""MathJax-Element-1-Frame",22,"id=""MathJax-Element-1-Frame",id='mathjax-element-1-fram
271947,r_1816,Beantwortung der Impulsfrage Mögliche Darstell...,"role=""presentation",24,"role=""presentation",role='presentation
273706,r_5135,Beantwortung der Impulsfrage Mögliche Darstell...,"role=""presentation",28,"role=""presentation",role='presentation
273702,r_5135,Beantwortung der Impulsfrage Mögliche Darstell...,"tabindex=""0",24,"tabindex=""0",tabindex='0


OK, and I think that those are in the format that I can export them.

In [24]:
with open("bea_3way_data.pickle", 'wb') as fOut:
    pickle.dump({'questions':questions_df, 'responses':responses_df, 'text':text_df.drop('spacy', axis='columns')}, fOut)